# Course 18: Responsible AI — Fairness & Bias

**GCP Professional Machine Learning Engineer**  
**Exam Section 6: Monitoring ML Solutions**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-mle/notebooks/18-responsible-ai-fairness.ipynb)

This notebook covers:
1. Understanding and measuring bias in datasets
2. Fairness metrics: demographic parity, equalized odds, calibration
3. Sliced model evaluation for subgroups
4. Bias mitigation strategies (pre-, in-, post-processing)
5. What-If Tool for interactive fairness exploration
6. Fairness Indicators with TensorFlow Model Analysis
7. Vertex AI Model Evaluation for fairness
8. Building a fairness evaluation pipeline

---
## Setup & Installation

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("All imports successful!")

---
## 1. Understanding Bias in Datasets

We'll create a synthetic lending dataset that contains realistic biases to study. This simulates a common exam scenario: a model trained on biased historical data perpetuates unfair outcomes.

In [ ]:
def create_biased_lending_dataset(n=5000):
    """Create a synthetic lending dataset with embedded bias."""
    np.random.seed(42)
    
    # Demographics
    group = np.random.choice(['A', 'B'], size=n, p=[0.6, 0.4])
    age = np.where(group == 'A',
                   np.random.normal(45, 12, n),
                   np.random.normal(35, 10, n)).clip(18, 80).astype(int)
    
    # Financial features — Group B has systematically lower income (historical bias)
    income = np.where(group == 'A',
                      np.random.lognormal(11.0, 0.5, n),
                      np.random.lognormal(10.5, 0.6, n))
    
    credit_score = np.where(group == 'A',
                            np.random.normal(720, 60, n),
                            np.random.normal(680, 70, n)).clip(300, 850).astype(int)
    
    debt_ratio = np.random.uniform(0.05, 0.65, n)
    employment_years = np.random.exponential(5, n).clip(0, 30)
    loan_amount = np.random.lognormal(10, 0.8, n)
    
    # Label: loan approval (biased toward Group A)
    score = (
        0.3 * (credit_score - 600) / 200 +
        0.2 * np.log(income) / 12 +
        0.15 * (1 - debt_ratio) +
        0.1 * employment_years / 10 +
        0.15 * (group == 'A').astype(float)  # Direct bias!
    )
    noise = np.random.normal(0, 0.1, n)
    approved = (score + noise > 0.5).astype(int)
    
    df = pd.DataFrame({
        'group': group,
        'age': age,
        'income': income.round(2),
        'credit_score': credit_score,
        'debt_ratio': debt_ratio.round(3),
        'employment_years': employment_years.round(1),
        'loan_amount': loan_amount.round(2),
        'approved': approved
    })
    return df

df = create_biased_lending_dataset()
print(f"Dataset shape: {df.shape}")
print(f"\nApproval rates by group:")
print(df.groupby('group')['approved'].mean().round(4))
print(f"\nOverall approval rate: {df['approved'].mean():.4f}")
df.head()

In [ ]:
# Visualize the bias in the dataset
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Approval rates by group
approval_by_group = df.groupby('group')['approved'].mean()
axes[0].bar(approval_by_group.index, approval_by_group.values,
            color=['#3b82f6', '#ef4444'])
axes[0].set_title('Approval Rate by Group')
axes[0].set_ylabel('Approval Rate')
for i, (g, v) in enumerate(approval_by_group.items()):
    axes[0].text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')

# Credit score distributions
for g, color in zip(['A', 'B'], ['#3b82f6', '#ef4444']):
    subset = df[df['group'] == g]
    axes[1].hist(subset['credit_score'], bins=30, alpha=0.5,
                 label=f'Group {g}', color=color)
axes[1].set_title('Credit Score Distribution')
axes[1].legend()

# Income distributions
for g, color in zip(['A', 'B'], ['#3b82f6', '#ef4444']):
    subset = df[df['group'] == g]
    axes[2].hist(np.log10(subset['income']), bins=30, alpha=0.5,
                 label=f'Group {g}', color=color)
axes[2].set_title('Log Income Distribution')
axes[2].legend()

plt.suptitle('Dataset Bias Analysis', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 2. Fairness Metrics

Key fairness metrics the exam tests:
- **Demographic Parity**: P(ŷ=1|A) = P(ŷ=1|B) — equal approval rates
- **Equalized Odds**: P(ŷ=1|y=1,A) = P(ŷ=1|y=1,B) AND P(ŷ=1|y=0,A) = P(ŷ=1|y=0,B)
- **Equal Opportunity**: P(ŷ=1|y=1,A) = P(ŷ=1|y=1,B) — equal true positive rates
- **Calibration**: P(y=1|ŷ=p,A) = P(y=1|ŷ=p,B) — predictions mean the same thing for both groups
- **Disparate Impact Ratio**: P(ŷ=1|B) / P(ŷ=1|A) — 80% rule (should be > 0.8)

In [ ]:
# Train a model (including group as feature — to show the problem)
features = ['age', 'income', 'credit_score', 'debt_ratio',
            'employment_years', 'loan_amount']

X = df[features]
y = df['approved']
groups = df['group']

X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    X, y, groups, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Overall F1 Score: {f1_score(y_test, y_pred):.4f}")

In [ ]:
def compute_fairness_metrics(y_true, y_pred, y_prob, groups, privileged='A', unprivileged='B'):
    """Compute comprehensive fairness metrics."""
    results = {}
    
    for group_name in [privileged, unprivileged]:
        mask = groups == group_name
        yt = y_true[mask]
        yp = y_pred[mask]
        yprob = y_prob[mask]
        
        cm = confusion_matrix(yt, yp)
        tn, fp, fn, tp = cm.ravel()
        
        results[group_name] = {
            'n': mask.sum(),
            'positive_rate': yp.mean(),
            'base_rate': yt.mean(),
            'accuracy': accuracy_score(yt, yp),
            'tpr': tp / (tp + fn) if (tp + fn) > 0 else 0,  # True Positive Rate
            'fpr': fp / (fp + tn) if (fp + tn) > 0 else 0,  # False Positive Rate
            'precision': tp / (tp + fp) if (tp + fp) > 0 else 0,
            'fnr': fn / (fn + tp) if (fn + tp) > 0 else 0,  # False Negative Rate
            'avg_prob': yprob.mean(),
        }
    
    p = results[privileged]
    u = results[unprivileged]
    
    fairness = {
        'demographic_parity_diff': u['positive_rate'] - p['positive_rate'],
        'disparate_impact_ratio': u['positive_rate'] / p['positive_rate'] if p['positive_rate'] > 0 else float('inf'),
        'equal_opportunity_diff': u['tpr'] - p['tpr'],
        'equalized_odds_tpr_diff': u['tpr'] - p['tpr'],
        'equalized_odds_fpr_diff': u['fpr'] - p['fpr'],
        'predictive_parity_diff': u['precision'] - p['precision'],
    }
    
    return results, fairness


group_metrics, fairness = compute_fairness_metrics(
    y_test.values, y_pred, y_prob, g_test.values
)

print("Per-Group Performance:")
print("=" * 60)
for group, metrics in group_metrics.items():
    print(f"\nGroup {group} (n={metrics['n']})")
    print(f"  Positive Rate (predicted): {metrics['positive_rate']:.4f}")
    print(f"  Base Rate (actual):        {metrics['base_rate']:.4f}")
    print(f"  Accuracy:                  {metrics['accuracy']:.4f}")
    print(f"  TPR (Recall):              {metrics['tpr']:.4f}")
    print(f"  FPR:                       {metrics['fpr']:.4f}")
    print(f"  Precision:                 {metrics['precision']:.4f}")

print("\nFairness Metrics:")
print("=" * 60)
for metric, value in fairness.items():
    flag = '⚠️' if abs(value) > 0.1 or (metric == 'disparate_impact_ratio' and value < 0.8) else '✓'
    print(f"  {flag} {metric}: {value:.4f}")

print(f"\n80% Rule (Disparate Impact > 0.8): {'PASS' if fairness['disparate_impact_ratio'] >= 0.8 else 'FAIL'}")

In [ ]:
# Visualize fairness metrics
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Positive rate comparison
groups_list = ['A', 'B']
pos_rates = [group_metrics[g]['positive_rate'] for g in groups_list]
base_rates = [group_metrics[g]['base_rate'] for g in groups_list]
x = np.arange(len(groups_list))
axes[0].bar(x - 0.15, pos_rates, 0.3, label='Predicted', color='#3b82f6')
axes[0].bar(x + 0.15, base_rates, 0.3, label='Actual', color='#22c55e')
axes[0].set_xticks(x)
axes[0].set_xticklabels([f'Group {g}' for g in groups_list])
axes[0].set_title('Positive Rate: Predicted vs Actual')
axes[0].legend()
axes[0].set_ylabel('Rate')

# 2. TPR and FPR comparison
tprs = [group_metrics[g]['tpr'] for g in groups_list]
fprs = [group_metrics[g]['fpr'] for g in groups_list]
axes[1].bar(x - 0.15, tprs, 0.3, label='TPR', color='#22c55e')
axes[1].bar(x + 0.15, fprs, 0.3, label='FPR', color='#ef4444')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Group {g}' for g in groups_list])
axes[1].set_title('Equalized Odds: TPR & FPR')
axes[1].legend()

# 3. Fairness metrics summary
metric_names = ['Dem. Parity\nDiff', 'Equal Opp.\nDiff', 'Predictive\nParity Diff']
metric_values = [
    fairness['demographic_parity_diff'],
    fairness['equal_opportunity_diff'],
    fairness['predictive_parity_diff']
]
colors = ['#ef4444' if abs(v) > 0.1 else '#22c55e' for v in metric_values]
axes[2].bar(metric_names, metric_values, color=colors)
axes[2].axhline(y=0.1, color='red', linestyle='--', alpha=0.5, label='Threshold (0.1)')
axes[2].axhline(y=-0.1, color='red', linestyle='--', alpha=0.5)
axes[2].set_title('Fairness Metric Differences')
axes[2].set_ylabel('Difference (B - A)')
axes[2].legend()

plt.suptitle('Fairness Analysis Dashboard', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Sliced Model Evaluation

Slicing evaluates model performance across different subgroups — critical for identifying hidden biases the overall metrics miss.

In [ ]:
# Sliced evaluation by group AND other dimensions
test_df = X_test.copy()
test_df['group'] = g_test.values
test_df['y_true'] = y_test.values
test_df['y_pred'] = y_pred
test_df['y_prob'] = y_prob

# Create age bins
test_df['age_bin'] = pd.cut(test_df['age'], bins=[18, 30, 45, 60, 80],
                            labels=['18-30', '31-45', '46-60', '61-80'])

# Create income bins
test_df['income_bin'] = pd.qcut(test_df['income'], q=3,
                                labels=['Low', 'Medium', 'High'])

# Sliced evaluation function
def sliced_evaluation(df, slice_col):
    results = []
    for slice_val in df[slice_col].unique():
        mask = df[slice_col] == slice_val
        subset = df[mask]
        if len(subset) < 10:
            continue
        results.append({
            'slice': f"{slice_col}={slice_val}",
            'n': len(subset),
            'accuracy': accuracy_score(subset['y_true'], subset['y_pred']),
            'precision': precision_score(subset['y_true'], subset['y_pred'], zero_division=0),
            'recall': recall_score(subset['y_true'], subset['y_pred'], zero_division=0),
            'f1': f1_score(subset['y_true'], subset['y_pred'], zero_division=0),
            'approval_rate': subset['y_pred'].mean(),
        })
    return pd.DataFrame(results)

# Evaluate across multiple slices
for col in ['group', 'age_bin', 'income_bin']:
    print(f"\nSliced Evaluation by {col}:")
    print("=" * 80)
    result = sliced_evaluation(test_df, col)
    print(result.to_string(index=False))

In [ ]:
# Intersectional analysis: Group x Age
test_df['intersectional'] = test_df['group'] + '_' + test_df['age_bin'].astype(str)

print("Intersectional Analysis (Group x Age):")
print("=" * 80)
intersect_result = sliced_evaluation(test_df, 'intersectional')
intersect_result = intersect_result.sort_values('approval_rate')
print(intersect_result.to_string(index=False))

# Visualize intersectional approval rates
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#3b82f6' if 'A' in s else '#ef4444' for s in intersect_result['slice']]
ax.barh(intersect_result['slice'], intersect_result['approval_rate'], color=colors)
ax.set_xlabel('Predicted Approval Rate')
ax.set_title('Intersectional Fairness: Approval Rate by Group x Age')
ax.axvline(x=test_df['y_pred'].mean(), color='gray', linestyle='--',
           label=f'Overall avg: {test_df["y_pred"].mean():.3f}')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. Bias Mitigation Strategies

Three main approaches:
- **Pre-processing**: Fix the data before training (resampling, reweighting)
- **In-processing**: Modify the training algorithm (constrained optimization, adversarial debiasing)
- **Post-processing**: Adjust predictions after training (threshold tuning)

In [ ]:
# === Strategy 1: Pre-processing — Reweighting ===
# Give higher weights to underrepresented group-label combinations

def compute_sample_weights(groups, labels):
    """Compute reweighting factors for fairness."""
    n = len(groups)
    weights = np.ones(n)
    
    for g in np.unique(groups):
        for l in np.unique(labels):
            mask = (groups == g) & (labels == l)
            expected = (groups == g).mean() * (labels == l).mean()
            observed = mask.mean()
            if observed > 0:
                weights[mask] = expected / observed
    
    return weights

sample_weights = compute_sample_weights(g_train.values, y_train.values)

model_reweighted = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
model_reweighted.fit(X_train_s, y_train, sample_weight=sample_weights)

y_pred_rw = model_reweighted.predict(X_test_s)
y_prob_rw = model_reweighted.predict_proba(X_test_s)[:, 1]

_, fairness_rw = compute_fairness_metrics(y_test.values, y_pred_rw, y_prob_rw, g_test.values)

print("Reweighted Model Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_rw):.4f}")
print(f"  Dem. Parity Diff: {fairness_rw['demographic_parity_diff']:.4f}")
print(f"  Disparate Impact: {fairness_rw['disparate_impact_ratio']:.4f}")
print(f"  Equal Opp. Diff:  {fairness_rw['equal_opportunity_diff']:.4f}")

In [ ]:
# === Strategy 2: Post-processing — Group-specific Thresholds ===
# Find thresholds that equalize true positive rates (Equal Opportunity)

def find_equal_opportunity_thresholds(y_true, y_prob, groups, target_tpr=0.85):
    """Find per-group thresholds to achieve equal TPR."""
    thresholds = {}
    for g in np.unique(groups):
        mask = groups == g
        yt = y_true[mask]
        yp = y_prob[mask]
        
        # Binary search for threshold that achieves target TPR
        best_thresh = 0.5
        best_diff = float('inf')
        for t in np.arange(0.1, 0.9, 0.01):
            pred = (yp >= t).astype(int)
            tp = ((pred == 1) & (yt == 1)).sum()
            fn = ((pred == 0) & (yt == 1)).sum()
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
            diff = abs(tpr - target_tpr)
            if diff < best_diff:
                best_diff = diff
                best_thresh = t
        thresholds[g] = best_thresh
    return thresholds

eq_thresholds = find_equal_opportunity_thresholds(
    y_test.values, y_prob, g_test.values, target_tpr=0.85
)
print(f"Group-specific thresholds: {eq_thresholds}")

# Apply group-specific thresholds
y_pred_eq = np.zeros_like(y_pred)
for g, thresh in eq_thresholds.items():
    mask = g_test.values == g
    y_pred_eq[mask] = (y_prob[mask] >= thresh).astype(int)

_, fairness_eq = compute_fairness_metrics(y_test.values, y_pred_eq, y_prob, g_test.values)

print(f"\nEqual Opportunity Adjusted Results:")
print(f"  Accuracy: {accuracy_score(y_test, y_pred_eq):.4f}")
print(f"  Dem. Parity Diff: {fairness_eq['demographic_parity_diff']:.4f}")
print(f"  Equal Opp. Diff:  {fairness_eq['equal_opportunity_diff']:.4f}")

In [ ]:
# === Compare all approaches ===
comparison = pd.DataFrame({
    'Approach': ['Original', 'Reweighted', 'Threshold-Adjusted'],
    'Accuracy': [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_rw),
        accuracy_score(y_test, y_pred_eq)
    ],
    'Dem. Parity Diff': [
        fairness['demographic_parity_diff'],
        fairness_rw['demographic_parity_diff'],
        fairness_eq['demographic_parity_diff']
    ],
    'Disparate Impact': [
        fairness['disparate_impact_ratio'],
        fairness_rw['disparate_impact_ratio'],
        fairness_eq['disparate_impact_ratio']
    ],
    'Equal Opp. Diff': [
        fairness['equal_opportunity_diff'],
        fairness_rw['equal_opportunity_diff'],
        fairness_eq['equal_opportunity_diff']
    ],
})

print("Mitigation Strategy Comparison:")
print("=" * 80)
print(comparison.to_string(index=False))
print("\nNote: Lower absolute Diff values = fairer. Disparate Impact closer to 1.0 = fairer.")

---
## 5. What-If Tool Concepts

The What-If Tool (WIT) is Google's interactive visual tool for exploring ML model behavior. While it runs in a notebook widget, we'll demonstrate the core concepts it provides.

In [ ]:
# === What-If Tool Concepts: Counterfactual Analysis ===
# Key WIT feature: "What if this feature value changed?"

def counterfactual_analysis(model, scaler, sample, feature_names, feature_to_change, new_values):
    """Show how prediction changes when one feature varies."""
    results = []
    original = sample.copy()
    
    for val in new_values:
        modified = original.copy()
        modified[feature_to_change] = val
        modified_scaled = scaler.transform(modified.values.reshape(1, -1))
        prob = model.predict_proba(modified_scaled)[0][1]
        results.append({'value': val, 'approval_prob': prob})
    
    return pd.DataFrame(results)

# Pick a borderline sample from Group B
borderline_mask = (g_test.values == 'B') & (y_prob > 0.3) & (y_prob < 0.6)
if borderline_mask.sum() > 0:
    sample_idx = np.where(borderline_mask)[0][0]
    sample = X_test.iloc[sample_idx]
    
    print(f"Original sample (Group B):")
    print(f"  Credit Score: {sample['credit_score']:.0f}")
    print(f"  Income: ${sample['income']:,.0f}")
    print(f"  Approval Probability: {y_prob[sample_idx]:.4f}")
    
    # What if credit score changed?
    credit_range = np.arange(500, 850, 10)
    cf_results = counterfactual_analysis(
        model, scaler, sample, features, 'credit_score', credit_range
    )
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(cf_results['value'], cf_results['approval_prob'],
            color='#3b82f6', linewidth=2)
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Decision boundary')
    ax.axvline(x=sample['credit_score'], color='green', linestyle='--',
               alpha=0.5, label=f'Current ({sample["credit_score"]:.0f})')
    ax.fill_between(cf_results['value'], cf_results['approval_prob'],
                    alpha=0.1, color='#3b82f6')
    ax.set_xlabel('Credit Score')
    ax.set_ylabel('Approval Probability')
    ax.set_title('What-If Analysis: Credit Score Impact on Approval')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No borderline samples found — adjusting search criteria.")

In [ ]:
# === What-If Tool Concept: Nearest Counterfactual ===
# Find the most similar approved applicant for each denied applicant

from sklearn.neighbors import NearestNeighbors

denied_mask = y_pred == 0
approved_mask = y_pred == 1

if denied_mask.sum() > 0 and approved_mask.sum() > 0:
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(X_test_s[approved_mask])
    
    # Find nearest approved neighbor for first 5 denied applicants
    denied_indices = np.where(denied_mask)[0][:5]
    
    print("Nearest Counterfactual Analysis:")
    print("(For each denied applicant, find the most similar approved one)")
    print("=" * 70)
    
    for idx in denied_indices:
        distances, neighbors = nn.kneighbors(X_test_s[idx].reshape(1, -1))
        approved_indices = np.where(approved_mask)[0]
        nearest_approved = approved_indices[neighbors[0][0]]
        
        denied_sample = X_test.iloc[idx]
        approved_sample = X_test.iloc[nearest_approved]
        
        print(f"\nDenied #{idx} (Group {g_test.values[idx]}, prob={y_prob[idx]:.3f}) → "
              f"Nearest Approved (Group {g_test.values[nearest_approved]}, prob={y_prob[nearest_approved]:.3f})")
        
        # Show key differences
        for feat in features:
            diff = approved_sample[feat] - denied_sample[feat]
            if abs(diff) > 0.01:
                direction = '↑' if diff > 0 else '↓'
                print(f"  {feat}: {denied_sample[feat]:.1f} → {approved_sample[feat]:.1f} ({direction}{abs(diff):.1f})")

---
## 6. Fairness Indicators with TFMA Concepts

TensorFlow Model Analysis (TFMA) provides Fairness Indicators for large-scale model evaluation. We'll replicate the core concepts using sklearn.

In [ ]:
# === Fairness Indicators: Threshold Sweep ===
# How do fairness metrics change as we vary the decision threshold?

thresholds = np.arange(0.1, 0.9, 0.05)
sweep_results = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    _, fair_t = compute_fairness_metrics(y_test.values, y_pred_t, y_prob, g_test.values)
    sweep_results.append({
        'threshold': t,
        'accuracy': accuracy_score(y_test, y_pred_t),
        'dem_parity_diff': fair_t['demographic_parity_diff'],
        'disparate_impact': fair_t['disparate_impact_ratio'],
        'equal_opp_diff': fair_t['equal_opportunity_diff'],
    })

sweep_df = pd.DataFrame(sweep_results)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(sweep_df['threshold'], sweep_df['accuracy'], 'b-', linewidth=2)
axes[0].set_title('Accuracy vs Threshold')
axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Accuracy')

axes[1].plot(sweep_df['threshold'], sweep_df['dem_parity_diff'], 'r-', linewidth=2)
axes[1].axhline(y=0, color='gray', linestyle='--')
axes[1].fill_between(sweep_df['threshold'], -0.1, 0.1, alpha=0.1, color='green', label='Fair zone')
axes[1].set_title('Demographic Parity Diff vs Threshold')
axes[1].set_xlabel('Threshold')
axes[1].legend()

axes[2].plot(sweep_df['threshold'], sweep_df['disparate_impact'], 'g-', linewidth=2)
axes[2].axhline(y=0.8, color='red', linestyle='--', label='80% Rule')
axes[2].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_title('Disparate Impact vs Threshold')
axes[2].set_xlabel('Threshold')
axes[2].legend()

plt.suptitle('Fairness Indicators: Threshold Sweep', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Find optimal threshold
fair_zone = sweep_df[
    (sweep_df['disparate_impact'] >= 0.8) &
    (sweep_df['dem_parity_diff'].abs() <= 0.1)
]
if len(fair_zone) > 0:
    best = fair_zone.loc[fair_zone['accuracy'].idxmax()]
    print(f"\nBest fair threshold: {best['threshold']:.2f}")
    print(f"  Accuracy: {best['accuracy']:.4f}")
    print(f"  Disparate Impact: {best['disparate_impact']:.4f}")
else:
    print("\nNo threshold satisfies both fairness constraints — need model-level mitigation.")

In [ ]:
# === Calibration Analysis by Group ===
# Are predicted probabilities equally meaningful across groups?

def calibration_by_group(y_true, y_prob, groups, n_bins=10):
    """Compute calibration curves per group."""
    fig, ax = plt.subplots(figsize=(8, 6))
    
    for g, color in zip(np.unique(groups), ['#3b82f6', '#ef4444']):
        mask = groups == g
        yt = y_true[mask]
        yp = y_prob[mask]
        
        bin_edges = np.linspace(0, 1, n_bins + 1)
        bin_centers = []
        bin_means = []
        
        for i in range(n_bins):
            bin_mask = (yp >= bin_edges[i]) & (yp < bin_edges[i + 1])
            if bin_mask.sum() >= 5:
                bin_centers.append(yp[bin_mask].mean())
                bin_means.append(yt[bin_mask].mean())
        
        ax.plot(bin_centers, bin_means, 'o-', color=color,
                label=f'Group {g}', linewidth=2, markersize=6)
    
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    ax.set_xlabel('Mean Predicted Probability')
    ax.set_ylabel('Actual Positive Rate')
    ax.set_title('Calibration Curves by Group')
    ax.legend()
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    plt.tight_layout()
    plt.show()

calibration_by_group(y_test.values, y_prob, g_test.values)
print("If curves diverge, the model is not equally calibrated across groups.")
print("A probability of 0.7 should mean 70% chance of approval for BOTH groups.")

---
## 7. Vertex AI Model Evaluation for Fairness

Shows how to configure sliced evaluation on Vertex AI.  
**Note**: API calls are commented out to avoid costs.

In [ ]:
import os
PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")
REGION = "us-central1"

# ============================================================
# Vertex AI Sliced Evaluation — SDK Reference
# ============================================================
# REQUIRES: GCP project with billing enabled
# Uncomment to run — will incur charges

# from google.cloud import aiplatform
#
# aiplatform.init(project=PROJECT_ID, location=REGION)
#
# # After uploading & evaluating a model, retrieve sliced metrics:
# model = aiplatform.Model(model_name="projects/.../models/...")
#
# # Get model evaluation
# evaluations = model.list_model_evaluations()
# for evaluation in evaluations:
#     print(f"Evaluation: {evaluation.name}")
#     print(f"Metrics: {evaluation.metrics}")
#
#     # Get sliced evaluations
#     slices = evaluation.list_model_evaluation_slices()
#     for s in slices:
#         print(f"  Slice: {s.slice_.dimension} = {s.slice_.value}")
#         print(f"  Metrics: {s.metrics}")
#
# # AutoML models automatically generate sliced evaluations
# # for categorical features. For custom models, you need to
# # define slicing specs in your evaluation pipeline.
#
# # Example: Create evaluation job with slicing
# from google.cloud.aiplatform_v1.types import ModelEvaluation
#
# eval_job = aiplatform.ModelEvaluationJob.submit(
#     model_name=model.resource_name,
#     prediction_type="classification",
#     target_field_name="approved",
#     slicing_specs=[
#         {"feature_map": {"group": {"type": "CATEGORICAL"}}},
#         {"feature_map": {"age_bin": {"type": "CATEGORICAL"}}},
#     ],
#     staging_bucket=f"gs://{PROJECT_ID}-staging",
# )

print("Vertex AI Model Evaluation code ready — uncomment to run on GCP.")
print("\nKey exam points:")
print("  - AutoML generates sliced evaluations automatically for categorical features")
print("  - Custom models need explicit slicing specs in evaluation pipelines")
print("  - Sliced evaluation helps identify fairness issues per subgroup")
print("  - Use Model Registry to track evaluation results across versions")

---
## 8. Building a Fairness Evaluation Pipeline

A complete fairness evaluation report that you could integrate into a CI/CD pipeline.

In [ ]:
def fairness_evaluation_report(y_true, y_pred, y_prob, sensitive_features,
                               feature_name='group', privileged='A',
                               di_threshold=0.8, dp_threshold=0.1):
    """Generate a complete fairness evaluation report."""
    
    group_metrics, fairness_metrics = compute_fairness_metrics(
        y_true, y_pred, y_prob, sensitive_features, privileged
    )
    
    # Determine pass/fail
    checks = {
        'Disparate Impact (>0.8)': (
            fairness_metrics['disparate_impact_ratio'] >= di_threshold,
            fairness_metrics['disparate_impact_ratio']
        ),
        'Demographic Parity (|diff|<0.1)': (
            abs(fairness_metrics['demographic_parity_diff']) < dp_threshold,
            fairness_metrics['demographic_parity_diff']
        ),
        'Equal Opportunity (|diff|<0.1)': (
            abs(fairness_metrics['equal_opportunity_diff']) < dp_threshold,
            fairness_metrics['equal_opportunity_diff']
        ),
        'Predictive Parity (|diff|<0.1)': (
            abs(fairness_metrics['predictive_parity_diff']) < dp_threshold,
            fairness_metrics['predictive_parity_diff']
        ),
    }
    
    report = []
    report.append("=" * 60)
    report.append("FAIRNESS EVALUATION REPORT")
    report.append("=" * 60)
    report.append(f"Sensitive Feature: {feature_name}")
    report.append(f"Privileged Group: {privileged}")
    report.append(f"Date: 2026-03-06")
    report.append("")
    
    report.append("FAIRNESS CHECKS:")
    report.append("-" * 40)
    all_pass = True
    for check_name, (passed, value) in checks.items():
        status = 'PASS ✓' if passed else 'FAIL ✗'
        if not passed:
            all_pass = False
        report.append(f"  [{status}] {check_name}: {value:.4f}")
    
    report.append("")
    report.append(f"OVERALL: {'PASS — Model meets fairness criteria' if all_pass else 'FAIL — Fairness issues detected'}")
    
    if not all_pass:
        report.append("")
        report.append("RECOMMENDED ACTIONS:")
        report.append("  1. Review training data for representation bias")
        report.append("  2. Consider reweighting or resampling strategies")
        report.append("  3. Evaluate group-specific threshold adjustment")
        report.append("  4. Conduct intersectional analysis")
        report.append("  5. Engage stakeholders before deployment")
    
    report.append("=" * 60)
    return '\n'.join(report)

# Run the report
report = fairness_evaluation_report(
    y_test.values, y_pred, y_prob, g_test.values
)
print(report)

In [ ]:
# Save fairness report
with open("fairness_report.txt", "w") as f:
    f.write(report)
print("Fairness report saved to 'fairness_report.txt'")
print("\nIn a real pipeline, you would:")
print("  1. Run this as a Vertex AI Pipeline component")
print("  2. Store results in ML Metadata")
print("  3. Gate deployment on fairness checks passing")
print("  4. Track metrics over time for drift detection")

---
## Summary

| Concept | Description | Exam Relevance |
|---------|-------------|----------------|
| Demographic Parity | Equal positive prediction rates across groups | Know when to use (hiring, lending) |
| Equalized Odds | Equal TPR and FPR across groups | Strongest fairness criterion |
| Equal Opportunity | Equal TPR only | When FPR differences are acceptable |
| Disparate Impact | Ratio of positive rates (80% rule) | Legal compliance |
| Calibration | Predictions mean the same across groups | When probability values matter |
| Pre-processing | Fix data before training (reweighting) | Simplest approach |
| Post-processing | Adjust thresholds after training | No retraining needed |
| What-If Tool | Interactive bias exploration | Know capabilities |
| Fairness Indicators | Large-scale sliced evaluation | TFMA integration |

**Exam Tips:**
- No single fairness metric works for all scenarios — the choice depends on context
- Demographic parity and equalized odds are often in tension (can't satisfy both)
- The 80% disparate impact rule is a legal threshold, not a statistical one
- Always check intersectional fairness (e.g., Group x Age), not just individual attributes
- On Vertex AI, AutoML provides sliced evaluation automatically; custom models need explicit configuration